# 03 Ridge 학습·검증·보고서

학습과 성능 검증을 별도 셀로 나눕니다. **현재는 구현 단계**이며 실행 플래그를False로 둡니다. 다음 단계에서 학습을 먼저 실행하고 모델을 고정한 뒤 평가합니다.

MAPE가 낮을수록 수명 예측이 더 정확합니다. Gap은 Valid−Train, Test−Valid, Test−9.1로 계산합니다.


In [3]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, Image, Code
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/pipeline.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.pipeline import load_config, choose_raw_dir
CONFIG = load_config(ROOT / 'config/model.json')
RAW_DIR = choose_raw_dir(ROOT, CONFIG)
print('프로젝트:', ROOT)
print('원본 폴더:', RAW_DIR)


프로젝트: .
원본 폴더: ../archive


## 1 특징 CSV와 고정 그룹 분할

원본이 없어도 동봉 특징 CSV로 이 단계부터 실행할 수 있습니다. B1은 같은 충전 프로토콜이 학습/holdout 양쪽에 들어가지 않게 분리합니다.


In [4]:
%pip install scikit-learn

python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from src.pipeline import read_feature_csv
from src.splits import make_splits
frame = read_feature_csv(ROOT / 'data/processed/cell_features.csv', CONFIG)
manifest = make_splits(frame, CONFIG)
display(manifest.groupby(['batch', 'role']).size().rename('n'))


batch  role   
B1     train      35
       valid      11
B2     test_b2    39
B3     test_b3    44
Name: n, dtype: int64

## 2 모델 학습과 내부 CV 선택

기본 2개/추가 3개 × alpha 7개 × 그룹 5fold = 70fit, 최종 35개 학습 1fit입니다. 각 fold 학습 부분만으로 표준화합니다. 목표는 log10 수명, 점수는 역변환한 원래 수명 단위의 MAPE입니다. 선택 동률은 입력이 적은 구성, 그 안에서는 큰 alpha 순입니다.


In [9]:
RUN_TRAINING = True
from src.train import train_model
if RUN_TRAINING:
    selected = train_model(frame, CONFIG, ROOT / 'results', ROOT / 'models')
    print('선택 특징:', selected['selected_features'], 'alpha:', selected['alpha'])
    display(pd.read_csv(ROOT / 'results/cv_search_results.csv'))
else:
    print('학습 미실행: 다음 단계에 RUN_TRAINING=True로 실행합니다.')


선택 특징: ['dq100_10_log_variance', 'cycle10_rms_c_timeweighted'] alpha: 1


,feature_set,feature_count,alpha,features,n_cv,MAPE_percent,MAE_cycles,RMSE_cycles
0,baseline_2,2,1.000,dq100_10_log_variance;cycle10_rms_c_timeweighted,35,6.986528,58.388934,75.104226
1,baseline_2,2,0.100,dq100_10_log_variance;cycle10_rms_c_timeweighted,35,7.095557,59.527372,76.695680
2,baseline_2,2,0.010,dq100_10_log_variance;cycle10_rms_c_timeweighted,35,7.107454,59.654671,76.890277
3,baseline_2,2,0.001,dq100_10_log_variance;cycle10_rms_c_timeweighted,35,7.108654,59.667558,76.910175
4,recommended_3,3,0.100,dq100_10_log_variance;cycle10_rms_c_timeweight...,35,7.113677,59.639793,76.925624
5,recommended_3,3,1.000,dq100_10_log_variance;cycle10_rms_c_timeweight...,35,7.127374,59.174988,75.429050
6,recommended_3,3,0.010,dq100_10_log_variance;cycle10_rms_c_timeweight...,35,7.130968,59.796305,77.120552
7,recommended_3,3,0.001,dq100_10_log_variance;cycle10_rms_c_timeweight...,35,7.132710,59.812112,77.140608
8,baseline_2,2,10.000,dq100_10_log_variance;cycle10_rms_c_timeweighted,35,7.291869,59.380380,73.361275
9,recommended_3,3,10.000,dq100_10_log_variance;cycle10_rms_c_timeweight...,35,7.739148,61.898235,75.172736


## 3 선택 완료 후 동일 모델로 검증·평가

B1 holdout 11개/B2 39개를 평가합니다. B3 44개는 선택 사항입니다. 평가에서 다시 fit하거나 변수·alpha를 고르지 않습니다. 한 번 끝낸 평가 결과를 덮어쓰지 않습니다.


In [10]:
RUN_EVALUATION = True
INCLUDE_B3 = True  # 추가 평가까지 하려면 첫 평가 전에 True
from src.evaluate import evaluate_model
if RUN_EVALUATION:
    evaluation = evaluate_model(frame, ROOT / 'models', ROOT / 'results', INCLUDE_B3)
    display(evaluation['performance'][['구분', 'MAPE (%)', '비고']])
else:
    print('성능 검증 미실행: 저장 모델 선택을 마친 다음 평가합니다.')


,구분,MAPE (%),비고
0,Train (Batch 1 CV),6.986528,선택 후보의 그룹 CV; 낮을수록 예측이 더 정확함
1,Valid (Batch 1 Hold-out),16.153492,B1 holdout; 동일 저장 모델
2,Test (Batch 2),33.560841,필수 최종 배치 평가
3,Gap (Train-Valid),9.166964,Valid−Train (%p); (+) 과적합/집단 차이 검토
4,Gap (Valid-Test),17.407348,Test−Valid (%p); (+) 배치 간 일반화 저하 검토
5,Gap (Target-Test),24.460841,Test−9.1 (%p); 논문 Target과 다른 실험 구성
6,Test (Batch 3),13.742185,동일 모델 추가 평가
7,Gap (Batch2-Batch3),-19.818656,B3−B2 (%p); (+) B3에서 오차 증가
8,Gap (Target-Test) [Batch 3],4.642185,B3−9.1 (%p); 추가 참고 비교


## 4 성능 보고서와 오류 분석

주 지표 MAPE는 평균 상대 오차, MAE는 평균 cycle 오차, RMSE는 큰 오차를 보여줍니다. 예를 들어 Train 8%/Valid 12%이면 Gap=12−8=+4%p입니다. 평가셋별 APE 상위 3개 오류 사례와 실제/예측·상대 오차·입력 분포 차이 그림을 만듭니다.


In [11]:
RUN_REPORT = True
from src.report import load_evaluation, write_reports
if RUN_REPORT:
    saved = load_evaluation(frame, ROOT / 'models', ROOT / 'results')
    write_reports(saved, frame, ROOT / 'results')
    display(Markdown((ROOT / 'results/performance_report.md').read_text()))
else:
    print('보고서 생성 미실행: 평가 결과 저장 이후 실행합니다.')


# 성능 결과

MAPE가 낮을수록 수명 예측이 더 정확합니다. Gap의 단위는%p이며 양수는 뒤 평가의 오차 증가입니다.

| 구분 | MAPE (%) | 비고 |
| --- | --- | --- |
| Train (Batch 1 CV) | 6.987 | 선택 후보의 그룹 CV; 낮을수록 예측이 더 정확함 |
| Valid (Batch 1 Hold-out) | 16.153 | B1 holdout; 동일 저장 모델 |
| Test (Batch 2) | 33.561 | 필수 최종 배치 평가 |
| Gap (Train-Valid) | 9.167 | Valid−Train (%p); (+) 과적합/집단 차이 검토 |
| Gap (Valid-Test) | 17.407 | Test−Valid (%p); (+) 배치 간 일반화 저하 검토 |
| Gap (Target-Test) | 24.461 | Test−9.1 (%p); 논문 Target과 다른 실험 구성 |
| Test (Batch 3) | 13.742 | 동일 모델 추가 평가 |
| Gap (Batch2-Batch3) | -19.819 | B3−B2 (%p); (+) B3에서 오차 증가 |
| Gap (Target-Test) [Batch 3] | 4.642 | B3−9.1 (%p); 추가 참고 비교 |

## 보조 지표와 중앙 수명 기준 모델

| scope | model | MAPE_percent | MAE_cycles | RMSE_cycles | n |
| --- | --- | --- | --- | --- | --- |
| train_cv | Ridge | 6.987 | 58.389 | 75.104 | 35 |
| valid | Ridge | 16.153 | 165.652 | 234.920 | 11 |
| test_b2 | Ridge | 33.561 | 166.675 | 183.679 | 39 |
| test_b3 | Ridge | 13.742 | 173.457 | 269.477 | 44 |
| train_cv | Training median | 20.244 | 154.043 | 190.216 | 35 |
| valid | Training median | 16.488 | 154.818 | 192.923 | 11 |
| test_b2 | Training median | 72.400 | 341.051 | 364.606 | 39 |
| test_b3 | Training median | 20.232 | 251.341 | 370.602 | 44 |

MAE는 평균 사이클 오차, RMSE는 큰 오차의 영향을 보여줍니다. 중앙 수명 기준값은 CV에서 각 fold 학습 정답의 중앙값, Valid/B2/B3에서 학습35개 정답의 중앙값입니다.

## 해석 범위

Train은 모델 선택에 사용한 그룹 CV 점수이며 재대입 성능이 아닙니다. B2/B3는 DAY1 EDA에서 확인했으므로 완전히 미관측한 개발 외부 자료는 아닙니다. 논문9.1%는 과제 참고 Target이며, 이번 코호트·분할·라벨 규칙으로 정확한 논문 재현을 주장하지 않습니다.

![실제값과 예측값](figures/01_actual_vs_predicted.png)

![부호 있는 상대 오차](figures/02_signed_error.png)

![입력 특징 분포 차이](figures/03_feature_distribution_shift.png)

[큰 오차 사례](error_analysis.md)


## 해석 범위

Train은 선택에 사용한 CV 점수입니다. B2/B3는 DAY1 EDA에 쓰였고 라벨 종료 규칙도 배치별 차이가 있습니다. 논문 9.1%는 과제 Target으로 비교하며 정확한 동일 실험 재현이 아닙니다. 현장 ESS 적용에는 calendar aging·온도/SOC·팩 차이·불확실성과 외부 데이터 검증이 필요합니다.
